# Elementary and Core Backend Sweep

This notebook benchmarks arbPlusJAX core interval kernels and elementary helpers against the backends that are available on the current machine.

It separates JAX first-compile time from steady-state execution and includes an optional diagnostics path for JAXPR, HLO, and profiler traces.

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "arbplusjax").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repo root")

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "experiments" / "benchmarks"))

from arbplusjax import jax_diagnostics
import elementary_core_benchmark_support as bench_support

In [3]:
diagnostics = jax_diagnostics.JaxDiagnosticsConfig(
    enabled=False,
    capture_jaxpr=False,
    capture_hlo=False,
    trace_execution=False,
)

backend_status = bench_support.backend_status_frame()
bench_support.render_table(backend_status)

,backend,available,detail
0,arb_flint_c_ref,False,ctypes C reference loader
1,boost,False,no elementary/core boost adapter in this works...
2,mpmath,True,python module
3,scipy,True,python module
4,jax_scipy,True,python module
5,mathematica,False,wolframscript not found


In [ ]:
core_results = bench_support.run_core_function_sweep(
    sample_size=4096,
    repeats=8,
    accuracy_sample=128,
    diagnostics=diagnostics,
)
bench_support.render_table(bench_support.summarize_results(core_results))

In [ ]:
mode_results = bench_support.run_core_mode_sweep(
    sample_size=2048,
    repeats=8,
    diagnostics=diagnostics,
)
bench_support.render_table(mode_results)

In [ ]:
ad_results = bench_support.run_core_ad_sweep(
    sample_size=256,
    repeats=6,
    fd_eps=1e-6,
    diagnostics=diagnostics,
)
bench_support.render_table(ad_results)

In [ ]:
elementary_results = bench_support.run_elementary_function_sweep(
    sample_size=2048,
    repeats=8,
    accuracy_sample=128,
    diagnostics=diagnostics,
)
bench_support.render_table(bench_support.summarize_results(elementary_results))

In [ ]:
core_paths = bench_support.write_result_tables("core_backend_sweep", core_results)
mode_paths = bench_support.write_result_tables("core_mode_sweep", mode_results)
ad_paths = bench_support.write_result_tables("core_ad_sweep", ad_results)
elementary_paths = bench_support.write_result_tables("elementary_backend_sweep", elementary_results)
[
    {"table": "core", **{k: str(v) for k, v in core_paths.items()}},
    {"table": "modes", **{k: str(v) for k, v in mode_paths.items()}},
    {"table": "ad", **{k: str(v) for k, v in ad_paths.items()}},
    {"table": "elementary", **{k: str(v) for k, v in elementary_paths.items()}},
]